In [4]:
#Implèmenter  deux matrice  addition kernels 
from numba import cuda
import numpy as np
import time

@cuda.jit
def matrix_add_coalesced(a, b, out):
    x, y = cuda.grid(2)
    if y < a.shape[0] and x < a.shape[1]:
        out[y, x] = a[y, x] + b[y, x]

@cuda.jit
def matrix_add_uncoalesced(a, b, out):
    x, y = cuda.grid(2)
    if x < a.shape[0] and y < a.shape[1]:
        out[x, y] = a[x, y] + b[x, y]

print("Q1 : deux kernels CUDA définis avec succès.")
print("GPU CUDA disponible :", cuda.is_available())

Q1 : deux kernels CUDA définis avec succès.
GPU CUDA disponible : True


In [5]:
if not cuda.is_available():
    raise RuntimeError(
        "Aucun GPU NVIDIA CUDA utilisable. Vérifie le pilote NVIDIA "
        "et que le noyau VS Code utilise le bon environnement Python."
    )

N = 8192
A = np.ones((N, N), dtype=np.float32)
B = np.full((N, N), 2.0, dtype=np.float32)

# Transfert CPU (host) -> GPU (device)
d_a = cuda.to_device(A)
d_b = cuda.to_device(B)
d_out = cuda.device_array((N, N), dtype=np.float32)

print(f"Q2 : A et B = {N} x {N}, dtype = {A.dtype}")
print(f"Taille de A : {A.nbytes / (1024**2):.0f} Mio")
print(f"Taille de B : {B.nbytes / (1024**2):.0f} Mio")
print(f"Mémoire GPU minimale pour A, B et sortie : {3 * A.nbytes / (1024**2):.0f} Mio")
print("Transfert vers le GPU terminé.")

Q2 : A et B = 8192 x 8192, dtype = float32
Taille de A : 256 Mio
Taille de B : 256 Mio
Mémoire GPU minimale pour A, B et sortie : 768 Mio
Transfert vers le GPU terminé.


In [6]:
threadsperblock = (32, 8)
blockspergrid = (
    (N + threadsperblock[0] - 1) // threadsperblock[0],
    (N + threadsperblock[1] - 1) // threadsperblock[1]
)

print("Q3 : dimensions du bloc :", threadsperblock)
print("Threads par bloc :", threadsperblock[0] * threadsperblock[1])
print("Dimensions de la grille :", blockspergrid)
print("Nombre total de blocs :", blockspergrid[0] * blockspergrid[1])

Q3 : dimensions du bloc : (32, 8)
Threads par bloc : 256
Dimensions de la grille : (256, 1024)
Nombre total de blocs : 262144


In [ ]:
def benchmark_kernel(kernel, iterations=10):
    # Dry run : compilation JIT et première exécution, NON chronométrées
    kernel[blockspergrid, threadsperblock](d_a, d_b, d_out)
    cuda.synchronize()

    times = []
    for _ in range(iterations):
        start = time.perf_counter()
        kernel[blockspergrid, threadsperblock](d_a, d_b, d_out)
        cuda.synchronize()  # attendre que le kernel se termine réellement
        end = time.perf_counter()
        times.append(end - start)

    return sum(times) / len(times)  # moyenne en secondes


N_ITER = 10
time_coalesced = benchmark_kernel(matrix_add_coalesced, N_ITER)
time_uncoalesced = benchmark_kernel(matrix_add_uncoalesced, N_ITER)

print(f"Q4 : temps moyen coalesced   = {time_coalesced * 1000:.4f} ms")
print(f"Q4 : temps moyen uncoalesced = {time_uncoalesced * 1000:.4f} ms")
print(f"Nombre d'itérations mesurées par kernel : {N_ITER}")

Q4 : temps moyen coalesced   = 3.5112 ms
Q4 : temps moyen uncoalesced = 7.1716 ms
Nombre d'itérations mesurées par kernel : 10


In [8]:
speedup = time_uncoalesced / time_coalesced

print(f"Q5 : speedup mesuré = {speedup:.3f}x")
if speedup > 1:
    print(f"Le kernel coalesced est environ {speedup:.2f} fois plus rapide.")
elif speedup < 1:
    print("Sur cette mesure, le kernel coalesced n'est pas plus rapide.")
else:
    print("Les deux kernels ont le même temps moyen mesuré.")

Q5 : speedup mesuré = 2.042x
Le kernel coalesced est environ 2.04 fois plus rapide.


In [9]:
bytes_transferred = 3 * (N ** 2) * 4
bandwidth_coalesced = bytes_transferred / (time_coalesced * 1e9)
bandwidth_uncoalesced = bytes_transferred / (time_uncoalesced * 1e9)

print(f"Q6 : volume de données théorique = {bytes_transferred:,} octets")
print(f"Bande passante coalesced   = {bandwidth_coalesced:.3f} GB/s")
print(f"Bande passante uncoalesced = {bandwidth_uncoalesced:.3f} GB/s")

Q6 : volume de données théorique = 805,306,368 octets
Bande passante coalesced   = 229.355 GB/s
Bande passante uncoalesced = 112.292 GB/s


Why does accessing consecutive rows instead of columns slash throughput on the same GPU hardware? NumPy stocke les matrices en row-major. Dans matrix_add_coalesced, les threads voisins accèdent à des éléments contigus (a[y, x]), ce qui permet le memory coalescing et améliore le débit mémoire. Dans matrix_add_uncoalesced, les accès (a[x, y]) sont espacés en mémoire, ce qui génère plus de transactions et réduit généralement les performances. accès contigus = coalescing = plus rapide ; accès dispersés = moins efficace.